# Accident Detection - Training

In [ ]:
# Cross-Platform Environment Setup
import os
import sys

def setup_environment():
    """Detects platform and sets up the environment."""
    in_colab = 'google.colab' in sys.modules
    in_kaggle = os.environ.get('KAGGLE_URL_BASE') is not None
    
    if in_colab or in_kaggle:
        print(f"Running on {'Google Colab' if in_colab else 'Kaggle'}. Installing dependencies...")
        !pip install -qU ultralytics wandb roboflow python-dotenv supervision easyocr cvzone
        
        if in_colab:
            from google.colab import userdata
            os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
            os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
        elif in_kaggle:
            try:
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
                os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY')
            except Exception:
                print("Kaggle secrets not found. Please set them in the Add-ons menu.")
    else:
        print("Running locally. Loading environment...")
        try:
            from dotenv import load_dotenv
            load_dotenv('../.env')
        except ImportError:
            print("python-dotenv not found. Install it with: pip install python-dotenv")

    # Verify keys
    if not os.environ.get('WANDB_API_KEY'):
        print("Warning: WANDB_API_KEY not set.")
    if not os.environ.get('ROBOFLOW_API_KEY'):
        print("Warning: ROBOFLOW_API_KEY not set.")

setup_environment()

## 1. W&B & Dataset Setup

In [ ]:
import wandb
from roboflow import Roboflow
from ultralytics import YOLO

# Login to W&B
if os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ.get('WANDB_API_KEY'))

# Download Dataset
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("zihan-yv8sc").project("zihan.v5i.yolov8-dataset")
version = project.version(5)
dataset = version.download("yolov8", location='../datasets')

In [ ]:
# ── Training Config ──────────────
EPOCHS = 15
IMGSZ  = 640
BATCH  = 16
MODEL_TYPE = "yolov8n.pt"
PROJECT_NAME = "Accident_Detection_V1"
RUN_NAME = "v1_accident_detection"

run = wandb.init(
    entity="ahmed-hossam-suez-canal-university",
    project=PROJECT_NAME,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL_TYPE,
        "pretrained":   True,
        "epochs":       EPOCHS,
        "imgsz":        IMGSZ,
        "batch":        BATCH,
        "fraction":     1.0,
        "dataset":      "accident-detection",
    }
)
print(f"W&B run started: {run.url}")

## 2. Modeling

In [ ]:
import glob
cfg = wandb.config
model = YOLO(cfg.model)

results = model.train(
    project='../experiments/runs/detect',
    data=glob.glob('../datasets/zihan.v5i.yolov8-dataset/data.yaml')[0],
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    fraction=cfg.fraction,
    name=RUN_NAME,
    plots=True,
)

In [ ]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/box_loss":     metrics_dict.get("val/box_loss",         0),
    "final/cls_loss":     metrics_dict.get("val/cls_loss",         0),
    "final/dfl_loss":     metrics_dict.get("val/dfl_loss",         0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


In [ ]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name=f"{RUN_NAME}_model",
    type="model",
    description="YOLOv8n fine-tuned",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


In [ ]:
wandb.finish()